In [ ]:
import pandas as pd
import os
import glob
import re

In [2]:


STATS_CSV = "/FastHome/gyula/designs/place_holder/DATA/gy_5/mpnn_design_stats.csv"

RANKED_ACCEPTED = "/FastHome/gyula/designs/place_holder/DATA/gy_5/Accepted/Ranked/"
RANKED_REJECTED = "/FastHome/gyula/designs/place_holder/DATA/gy_5_rejected/Ranked/Ranked/"

MMPBSA_ACCEPTED = "/FastHome/gyula/designs/place_holder/DATA/gy_5/md_sim_analysis/mmpbsa/jobs/delta_total_summary.csv"
MMPBSA_REJECTED = "/FastHome/gyula/designs/place_holder/DATA/gy_5_rejected/md_sim_analysis/mmpbsa/jobs/delta_total_summary.csv"

OUTPUT_CSV = "/FastHome/gyula/designs/place_holder/DATA/gy_5/mpnn_design_stats_augmented.csv"


def norm_rank(x):
    m = re.search(r"(\d+)", str(x))
    return str(int(m.group(1))) if m else None


def parse_ranked_folder(folder, source):
    rows = []
    for pdb in glob.glob(os.path.join(folder, "*.pdb")):
        base = os.path.basename(pdb)
        m = re.match(r"^(\d+)_(.+?)_model\d+\.pdb$", base)
        if not m:
            continue
        rows.append({
            "rank_norm": norm_rank(m.group(1)),
            "rank_raw": m.group(1),
            "Design": m.group(2),
            "pdb_path": pdb,
            "found_in": source,
        })
    return pd.DataFrame(rows)


def load_mmpbsa(path, suffix):
    d = pd.read_csv(path).copy()
    d["job_norm"] = d["subdir"].astype(str).str.extract(r"job_(\d+)")[0].map(norm_rank)
    d = d.rename(columns={
        "delta_total": f"delta_total_{suffix}",
        "dG_binding_IE": f"dG_binding_IE_{suffix}",
        "dG_binding_IE_err": f"dG_binding_IE_err_{suffix}",
    })
    return d[[
        "job_norm",
        f"delta_total_{suffix}",
        f"dG_binding_IE_{suffix}",
        f"dG_binding_IE_err_{suffix}",
    ]].copy()


# ── 1. Load MPNN stats ────────────────────────────────────────────────────────
df = pd.read_csv(STATS_CSV).copy()
if "Design" not in df.columns:
    df = df.rename(columns={df.columns[0]: "Design"})

# ── 2. Parse Ranked folders separately ────────────────────────────────────────
acc_ranked = parse_ranked_folder(RANKED_ACCEPTED, "accepted")
rej_ranked = parse_ranked_folder(RANKED_REJECTED, "rejected")

# Index by rank within each branch
acc_ranked = acc_ranked.drop_duplicates(subset=["rank_norm"], keep="first")
rej_ranked = rej_ranked.drop_duplicates(subset=["rank_norm"], keep="first")

# ── 3. Load MMPBSA summaries ──────────────────────────────────────────────────
acc_mmpbsa = load_mmpbsa(MMPBSA_ACCEPTED, "accepted")
rej_mmpbsa = load_mmpbsa(MMPBSA_REJECTED, "rejected")

# ── 4. Map each job to the corresponding ranked PDB in the same branch ────────
acc_job_to_design = acc_mmpbsa.merge(
    acc_ranked,
    left_on="job_norm",
    right_on="rank_norm",
    how="left"
)[["job_norm", "Design"]].drop_duplicates(subset=["job_norm"], keep="first")

rej_job_to_design = rej_mmpbsa.merge(
    rej_ranked,
    left_on="job_norm",
    right_on="rank_norm",
    how="left"
)[["job_norm", "Design"]].drop_duplicates(subset=["job_norm"], keep="first")

# ── 5. Convert job mapping into design-keyed MMPBSA tables ────────────────────
acc_mmpbsa = acc_mmpbsa.merge(acc_job_to_design, on="job_norm", how="left")
rej_mmpbsa = rej_mmpbsa.merge(rej_job_to_design, on="job_norm", how="left")

acc_lookup = acc_mmpbsa.set_index("Design")[[
    "delta_total_accepted", "dG_binding_IE_accepted", "dG_binding_IE_err_accepted"
]]

rej_lookup = rej_mmpbsa.set_index("Design")[[
    "delta_total_rejected", "dG_binding_IE_rejected", "dG_binding_IE_err_rejected"
]]

# ── 6. Merge into the main MPNN dataframe by exact Design ─────────────────────
df = df.merge(acc_lookup, on="Design", how="left")
df = df.merge(rej_lookup, on="Design", how="left")

# ── 7. Final combined columns ────────────────────────────────────────────────
df["delta_total"] = df["delta_total_accepted"].combine_first(df["delta_total_rejected"])
df["dG_binding_IE"] = df["dG_binding_IE_accepted"].combine_first(df["dG_binding_IE_rejected"])
df["dG_binding_IE_err"] = df["dG_binding_IE_err_accepted"].combine_first(df["dG_binding_IE_err_rejected"])

# ── 8. Add accepted/rejected/both label from Ranked folders ───────────────────
acc_designs = set(acc_ranked["Design"].dropna())
rej_designs = set(rej_ranked["Design"].dropna())

def source_label(design):
    a = design in acc_designs
    r = design in rej_designs
    if a and r:
        return "both"
    if a:
        return "accepted"
    if r:
        return "rejected"
    return None

df["found_in"] = df["Design"].map(source_label)

# ── 9. Save ───────────────────────────────────────────────────────────────────
df = df.copy()
df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved → {OUTPUT_CSV}")
print(df[[
    "Design",
    "found_in",
    "delta_total_accepted",
    "delta_total_rejected",
    "delta_total",
    "dG_binding_IE_accepted",
    "dG_binding_IE_rejected",
    "dG_binding_IE",
]].head(10).to_string(index=False))

Saved → /FastHome/gyula/designs/place_holder/DATA/gy_5/mpnn_design_stats_augmented.csv
                    Design found_in  delta_total_accepted  delta_total_rejected  delta_total  dG_binding_IE_accepted  dG_binding_IE_rejected  dG_binding_IE
BAX_1F16_l21_s870652_mpnn1 accepted               -101.11                   NaN      -101.11                  -49.61                     NaN         -49.61
BAX_1F16_l21_s870652_mpnn2 accepted                -93.41                   NaN       -93.41                  -35.93                     NaN         -35.93
BAX_1F16_l21_s312141_mpnn1 rejected                   NaN                -81.83       -81.83                     NaN                   -6.78          -6.78
BAX_1F16_l21_s312141_mpnn2 accepted                -82.20                   NaN       -82.20                  -20.08                     NaN         -20.08
BAX_1F16_l21_s312141_mpnn3 accepted                -86.52                   NaN       -86.52                  -18.48                 

In [3]:
df

,Design,Protocol,Length,Seed,Helicity,Target_Hotspot,Sequence,InterfaceResidues,MPNN_score,MPNN_seq_recovery,...,delta_total_accepted,dG_binding_IE_accepted,dG_binding_IE_err_accepted,delta_total_rejected,dG_binding_IE_rejected,dG_binding_IE_err_rejected,delta_total,dG_binding_IE,dG_binding_IE_err,found_in
0,BAX_1F16_l21_s870652_mpnn1,3stage,21,870652,0.95,74-99,GRSEMMKRFMEIVNSTWESFR,"B2,B5,B6,B8,B9,B10,B12,B13,B14,B16,B17,B18,B20...",1.52,0.37,...,-101.11,-49.61,9.71,NaN,NaN,NaN,-101.11,-49.61,9.71,accepted
1,BAX_1F16_l21_s870652_mpnn2,3stage,21,870652,0.95,74-99,GRSEMMKRFMDIVNETWEKFR,"B2,B3,B5,B6,B8,B9,B10,B12,B13,B14,B16,B17,B18,...",1.54,0.25,...,-93.41,-35.93,12.57,NaN,NaN,NaN,-93.41,-35.93,12.57,accepted
2,BAX_1F16_l21_s312141_mpnn1,3stage,21,312141,0.95,74-99,DAKATISQVVATMMAQLDSLR,"B2,B3,B5,B6,B8,B9,B10,B12,B13,B14,B16,B17,B18,...",1.75,0.00,...,NaN,NaN,NaN,-81.83,-6.78,8.21,-81.83,-6.78,8.21,rejected
3,BAX_1F16_l21_s312141_mpnn2,3stage,21,312141,0.95,74-99,DAKKTISQVVETMMAQLDSLR,"B2,B3,B5,B6,B8,B9,B10,B12,B13,B14,B16,B17,B18,...",1.79,0.00,...,-82.20,-20.08,11.00,NaN,NaN,NaN,-82.20,-20.08,11.00,accepted
4,BAX_1F16_l21_s312141_mpnn3,3stage,21,312141,0.95,74-99,DAKKTISQVVDTMMAQLDSLR,"B2,B3,B5,B6,B8,B9,B10,B12,B13,B14,B16,B17,B18,...",1.82,0.17,...,-86.52,-18.48,7.88,NaN,NaN,NaN,-86.52,-18.48,7.88,accepted
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
555,BAX_1F16_l22_s711858_mpnn4,3stage,22,711858,0.95,74-99,SMEDVRNTLVGLMLSLIDEIMS,"B2,B3,B5,B6,B7,B8,B9,B10,B12,B13,B14,B16,B17,B...",1.31,0.22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
556,BAX_1F16_l22_s711858_mpnn5,3stage,22,711858,0.95,74-99,SMEDKRNTLVGLMLSLIDEIMK,"B5,B7,B8,B9,B10,B12,B13,B14,B16,B17,B18,B20,B21",1.32,0.22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
557,BAX_1F16_l22_s711858_mpnn6,3stage,22,711858,0.95,74-99,SMEDMRNTLVGLMLSLIDEIMA,"B2,B5,B6,B7,B8,B9,B10,B12,B13,B14,B16,B17,B18,...",1.32,0.33,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
558,BAX_1F16_l22_s711858_mpnn7,3stage,22,711858,0.95,74-99,SMEDMRNTLVGLMLSLIDEIMS,"B2,B5,B6,B7,B8,B9,B10,B12,B13,B14,B16,B17,B18,...",1.34,0.22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
print(1)

1


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

INPUT_CSV = "/FastHome/gyula/designs/place_holder/DATA/gy_5/mpnn_design_stats_augmented.csv"
OUTPUT_DIR = Path("/FastHome/gyula/designs/place_holder/DATA/gy_5/output")
OUTPUT_DIR.mkdir(exist_ok=True)
OUT_PDF = OUTPUT_DIR / "delta_total_correlations.pdf"
OUT_CSV = OUTPUT_DIR / "delta_total_correlations.csv"

# colorblind-friendly palette
COLORS = {
    "accepted": "#377eb8",
    "rejected": "#ff7f00",
    "both": "#4daf4a",
    "unknown": "#999999",
}

plt.style.use("seaborn-v0_8-whitegrid")

df = pd.read_csv(INPUT_CSV).copy()

if "delta_total" not in df.columns:
    raise ValueError("delta_total column not found. Build the merged dataframe first.")

if "found_in" not in df.columns:
    df["found_in"] = "unknown"
else:
    df["found_in"] = df["found_in"].fillna("unknown").astype(str)

# Keep only numeric columns, exclude duplicated source-specific total columns from the sweep if desired
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude = {
    "delta_total_accepted", "delta_total_rejected",
    "dG_binding_IE_accepted", "dG_binding_IE_rejected",
    "dG_binding_IE_err_accepted", "dG_binding_IE_err_rejected",
}
metrics = [c for c in numeric_cols if c != "delta_total" and c not in exclude]

rows = []

with PdfPages(OUT_PDF) as pdf:
    # overview page
    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.axis("off")
    accepted_n = (df["found_in"] == "accepted").sum()
    rejected_n = (df["found_in"] == "rejected").sum()
    both_n = (df["found_in"] == "both").sum()
    unknown_n = (df["found_in"] == "unknown").sum()
    text = f"delta_total vs every numeric metric\n\nRows: {len(df)}\nMetrics plotted: {len(metrics)}\nAccepted: {accepted_n}\nRejected: {rejected_n}\nBoth: {both_n}\nUnknown: {unknown_n}\n\nEach following page shows a scatter plot of delta_total vs one metric,\nwith Pearson r computed on non-missing pairs.\nColors are colorblind-friendly."
    ax.text(0.03, 0.97, text, va="top", ha="left", fontsize=16)
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    for metric in metrics:
        plot_df = df[["delta_total", metric, "found_in"]].dropna()
        n = len(plot_df)
        r = plot_df[["delta_total", metric]].corr(method="pearson").iloc[0, 1] if n >= 2 else np.nan
        rows.append({"metric": metric, "n": n, "pearson_r": r})

        fig, ax = plt.subplots(figsize=(10, 7))
        for grp in ["accepted", "rejected", "both", "unknown"]:
            sub = plot_df[plot_df["found_in"] == grp]
            if len(sub) == 0:
                continue
            ax.scatter(
                sub[metric], sub["delta_total"],
                s=30, alpha=0.75, c=COLORS[grp], label=f"{grp} (n={len(sub)})",
                edgecolors="none"
            )

        ax.set_title(f"delta_total vs {metric}\nPearson r = {r:.3f}   n = {n}")
        ax.set_xlabel(metric)
        ax.set_ylabel("delta_total")
        ax.legend(frameon=True)
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

corr_df = pd.DataFrame(rows).sort_values(by="pearson_r", key=lambda s: s.abs(), ascending=False)
corr_df.to_csv(OUT_CSV, index=False)
print(str(OUT_PDF))
print(str(OUT_CSV))

/tmp/ipykernel_3159215/666823803.py:77: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(frameon=True)
/tmp/ipykernel_3159215/666823803.py:77: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(frameon=True)
/tmp/ipykernel_3159215/666823803.py:77: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(frameon=True)
/tmp/ipykernel_3159215/666823803.py:77: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(frameon=True)
/tmp/ipykernel_3159215/666823803.py:77: UserWarning: No 

/FastHome/gyula/designs/place_holder/DATA/gy_5/output/delta_total_correlations.pdf
/FastHome/gyula/designs/place_holder/DATA/gy_5/output/delta_total_correlations.csv


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

INPUT_CSV = "/FastHome/gyula/designs/place_holder/DATA/gy_5/mpnn_design_stats_augmented.csv"
OUTPUT_DIR = Path("/FastHome/gyula/designs/place_holder/DATA/gy_5/output_new")
OUTPUT_DIR.mkdir(exist_ok=True)
OUT_PDF = OUTPUT_DIR / "delta_total_correlations.pdf"
OUT_CSV = OUTPUT_DIR / "delta_total_correlations.csv"



OUT_PDF = OUTPUT_DIR / "delta_total_correlations.pdf"
OUT_CSV = OUTPUT_DIR / "delta_total_correlations.csv"
ORDER = ["accepted", "rejected", "both", "unknown"]
COLORS = {
    "accepted": "#377eb8",
    "rejected": "#ff7f00",
    "both":     "#4daf4a",
    "unknown":  "#999999",
}

plt.style.use("seaborn-v0_8-whitegrid")

df = pd.read_csv(INPUT_CSV).copy()
if "Design" not in df.columns:
    df = df.rename(columns={df.columns[0]: "Design"})
if "found_in" not in df.columns:
    df["found_in"] = "unknown"
else:
    df["found_in"] = df["found_in"].fillna("unknown").astype(str)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude = {
    "delta_total_accepted", "delta_total_rejected",
    "dG_binding_IE_accepted", "dG_binding_IE_rejected",
    "dG_binding_IE_err_accepted", "dG_binding_IE_err_rejected",
}
metrics = [c for c in numeric_cols if c != "delta_total" and c not in exclude]

rows = []

with PdfPages(OUT_PDF) as pdf:

    # overview page
    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.axis("off")
    counts = df["found_in"].value_counts().to_dict()
    lines = [f"delta_total vs every numeric metric",
             f"",
             f"Total rows  : {len(df)}",
             f"Metrics     : {len(metrics)}",
             f"",
             *[f"  {k}: {v}" for k, v in counts.items()],
             f"",
             f"Scatter: individual points (faint)",
             f"Large marker + line: per-group mean"]
    ax.text(0.03, 0.97, "\n".join(lines), va="top", ha="left", fontsize=15,
            family="monospace")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    for metric in metrics:
        plot_df = df[["delta_total", metric, "found_in"]].dropna().copy()
        n = len(plot_df)
        if n < 2:
            continue

        r = plot_df[["delta_total", metric]].corr(method="pearson").iloc[0, 1]
        rows.append({"metric": metric, "n": n, "pearson_r": r})

        fig, ax = plt.subplots(figsize=(10, 7))

        # raw points
        for grp in ORDER:
            sub = plot_df[plot_df["found_in"] == grp]
            if sub.empty:
                continue

            ax.scatter(
                sub[metric],
                sub["delta_total"],
                s=18,
                alpha=0.20,
                color=COLORS[grp],
                edgecolors="none",
                zorder=1
            )

        # averaged points + connecting line per group
        for grp in ORDER:
            sub = plot_df[plot_df["found_in"] == grp]
            if sub.empty:
                continue

            mean_curve = (
                sub.groupby(metric, as_index=False)["delta_total"]
                   .mean()
                   .sort_values(by=metric)
            )

            ax.plot(
                mean_curve[metric],
                mean_curve["delta_total"],
                color=COLORS[grp],
                linewidth=2.0,
                alpha=0.95,
                zorder=3,
                label=f"{grp} mean"
            )

            ax.scatter(
                mean_curve[metric],
                mean_curve["delta_total"],
                s=45,
                color=COLORS[grp],
                edgecolors="black",
                linewidths=0.6,
                zorder=4
            )

        ax.set_title(f"delta_total vs {metric}\\nPearson r = {r:.3f}   n = {n}")
        ax.set_xlabel(metric)
        ax.set_ylabel("delta_total")
        ax.legend(frameon=True, fontsize=8, loc="best")
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

corr_df = (pd.DataFrame(rows)
           .dropna(subset=["pearson_r"])
           .sort_values(by="pearson_r", key=lambda s: s.abs(), ascending=False))
corr_df.to_csv(OUT_CSV, index=False)
print(f"Saved PDF  -> {OUT_PDF}")
print(f"Saved CSV  -> {OUT_CSV}")
print(corr_df.head(10).to_string(index=False))

Saved PDF  -> /FastHome/gyula/designs/place_holder/DATA/gy_5/output_new/delta_total_correlations.pdf
Saved CSV  -> /FastHome/gyula/designs/place_holder/DATA/gy_5/output_new/delta_total_correlations.csv
                     metric   n  pearson_r
              Average_dSASA 351  -0.886069
                    2_dSASA 351  -0.877851
                    1_dSASA 351  -0.876881
Average_n_InterfaceResidues 351  -0.823740
      2_n_InterfaceResidues 351  -0.815348
                     Length 351  -0.809461
      1_n_InterfaceResidues 351  -0.808591
                 Average_dG 351   0.761966
                       1_dG 351   0.734474
               2_Binder_pTM 351  -0.709114


In [7]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path

INPUT_CSV = "/FastHome/gyula/designs/place_holder/DATA/gy_5/mpnn_design_stats_augmented.csv"
OUTPUT_DIR = Path("/FastHome/gyula/designs/place_holder/DATA/gy_5/output_new")
OUTPUT_DIR.mkdir(exist_ok=True)
OUT_CSV = OUTPUT_DIR / "delta_total_linear_model_results.csv"

CANDIDATE_VARS = [
    "Length",
    "Helicity",
    "MPNN_score",
    "MPNN_seq_recovery",
    "Average_pLDDT",
    "Average_pTM",
    "Average_i_pTM",
    "Average_pAE",
    "Average_i_pAE",
    "Average_i_pLDDT",
    "Average_ss_pLDDT",
    "Average_Unrelaxed_Clashes",
    "Average_Relaxed_Clashes",
    "Average_Binder_Energy_Score",
    "Average_Surface_Hydrophobicity",
    "Average_ShapeComplementarity",
    "Average_PackStat",
    "Average_dG",
    "Average_dSASA",
    "Average_dG/dSASA",
    "Average_Interface_SASA_%",
    "Average_Interface_Hydrophobicity",
    "Average_n_InterfaceResidues",
    "Average_n_InterfaceHbonds",
    "Average_InterfaceHbondsPercentage",
    "Average_n_InterfaceUnsatHbonds",
    "Average_InterfaceUnsatHbondsPercentage",
    "Average_Interface_Helix%",
    "Average_Interface_BetaSheet%",
    "Average_Interface_Loop%",
    "Average_Binder_Helix%",
    "Average_Binder_BetaSheet%",
    "Average_Binder_Loop%",
    "Average_InterfaceAAs",
    "Average_Hotspot_RMSD",
    "Average_Target_RMSD",
    "Average_Binder_pLDDT",
    "Average_Binder_pTM",
    "Average_Binder_pAE",
    "Average_Binder_RMSD",
]

ALPHA = 0.05

df = pd.read_csv(INPUT_CSV).copy()

if "delta_total" not in df.columns:
    raise ValueError("delta_total column not found")

existing = [c for c in CANDIDATE_VARS if c in df.columns]
if "Length" not in existing:
    raise ValueError("Length column not found")

for c in ["delta_total"] + existing:
    df[c] = pd.to_numeric(df[c], errors="coerce")

results = []

base_df = df[["delta_total", "Length"]].dropna().copy()
X_base = sm.add_constant(base_df[["Length"]])
y_base = base_df["delta_total"]
base_model = sm.OLS(y_base, X_base).fit()
base_r2 = base_model.rsquared
base_aic = base_model.aic

for var in existing:
    if var == "Length":
        continue

    sub = df[["delta_total", "Length", var]].dropna().copy()
    if len(sub) < 5:
        continue

    m1 = sm.OLS(sub["delta_total"], sm.add_constant(sub[["Length"]])).fit()
    m2 = sm.OLS(sub["delta_total"], sm.add_constant(sub[["Length", var]])).fit()

    if int(m2.df_model) > int(m1.df_model):
        f_stat, p_nested, _ = m2.compare_f_test(m1)
    else:
        f_stat, p_nested, _ = np.nan, np.nan, np.nan

    coef = m2.params.get(var, np.nan)
    se = m2.bse.get(var, np.nan)
    pval = m2.pvalues.get(var, np.nan)
    ci_low, ci_high = m2.conf_int().loc[var].tolist()

    delta_r2 = m2.rsquared - m1.rsquared
    f2 = (delta_r2 / (1 - m2.rsquared)) if m2.rsquared < 1 else np.nan

    results.append({
        "variable": var,
        "n": int(m2.nobs),
        "length_only_r2": m1.rsquared,
        "length_plus_var_r2": m2.rsquared,
        "delta_r2": delta_r2,
        "f2_effect_size": f2,
        "coef": coef,
        "std_err": se,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": pval,
        "nested_model_p_value": p_nested,
        "nested_f_stat": f_stat,
        "aic_length_only": m1.aic,
        "aic_length_plus_var": m2.aic,
        "aic_improvement": m1.aic - m2.aic,
    })

res = pd.DataFrame(results)
if not res.empty:
    res = res.sort_values(by=["nested_model_p_value", "delta_r2"], ascending=[True, False])

res.to_csv(OUT_CSV, index=False)

better = res[(res["nested_model_p_value"].notna()) & (res["nested_model_p_value"] < ALPHA)]

print(f"Saved -> {OUT_CSV}")
print(f"Baseline Length-only model: R2={base_r2:.4f}, AIC={base_aic:.2f}")
print("\nTop variables that add explanatory power beyond Length:")
if better.empty:
    print("None at p < 0.05")
else:
    print(
        better[[
            "variable",
            "n",
            "delta_r2",
            "f2_effect_size",
            "coef",
            "std_err",
            "p_value",
            "nested_model_p_value",
            "aic_improvement",
        ]].head(20).to_string(index=False)
    )




Saved -> /FastHome/gyula/designs/place_holder/DATA/gy_5/output_new/delta_total_linear_model_results.csv
Baseline Length-only model: R2=0.6552, AIC=2503.23

Top variables that add explanatory power beyond Length:
                              variable   n  delta_r2  f2_effect_size       coef   std_err      p_value  nested_model_p_value  aic_improvement
                         Average_dSASA 351  0.131482        0.616445  -0.048472  0.003309 3.536429e-38          3.536429e-38       166.560548
              Average_Interface_SASA_% 351  0.049232        0.166582  -0.194210  0.025507 2.529827e-13          2.529827e-13        52.081318
           Average_n_InterfaceResidues 351  0.039255        0.128488  -2.510145  0.375386 9.099606e-11          9.099606e-11        40.428374
                            Average_dG 351  0.035769        0.115757   0.444879  0.070094 6.848373e-10          6.848373e-10        36.446200
             Average_n_InterfaceHbonds 351  0.022422        0.069558  -1.38658

In [8]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path

INPUT_CSV = "/FastHome/gyula/designs/place_holder/DATA/gy_5/mpnn_design_stats_augmented.csv"
OUTPUT_DIR = Path("/FastHome/gyula/designs/place_holder/DATA/gy_5/output_new")
OUTPUT_DIR.mkdir(exist_ok=True)
OUT_CSV = OUTPUT_DIR / "delta_total_linear_model_vs_length_dsasa.csv"

BASE_VARS = ["Length", "Average_dSASA"]

CANDIDATE_VARS = [
    "Helicity",
    "MPNN_score",
    "MPNN_seq_recovery",
    "Average_pLDDT",
    "Average_pTM",
    "Average_i_pTM",
    "Average_pAE",
    "Average_i_pAE",
    "Average_i_pLDDT",
    "Average_ss_pLDDT",
    "Average_Unrelaxed_Clashes",
    "Average_Relaxed_Clashes",
    "Average_Binder_Energy_Score",
    "Average_Surface_Hydrophobicity",
    "Average_ShapeComplementarity",
    "Average_PackStat",
    "Average_dG",
    "Average_dG/dSASA",
    "Average_Interface_SASA_%",
    "Average_Interface_Hydrophobicity",
    "Average_n_InterfaceResidues",
    "Average_n_InterfaceHbonds",
    "Average_InterfaceHbondsPercentage",
    "Average_n_InterfaceUnsatHbonds",
    "Average_InterfaceUnsatHbondsPercentage",
    "Average_Interface_Helix%",
    "Average_Interface_BetaSheet%",
    "Average_Interface_Loop%",
    "Average_Binder_Helix%",
    "Average_Binder_BetaSheet%",
    "Average_Binder_Loop%",
    "Average_Hotspot_RMSD",
    "Average_Target_RMSD",
    "Average_Binder_pLDDT",
    "Average_Binder_pTM",
    "Average_Binder_pAE",
    "Average_Binder_RMSD",
]

ALPHA = 0.05

df = pd.read_csv(INPUT_CSV).copy()

if "delta_total" not in df.columns:
    raise ValueError("delta_total column not found")

needed = ["delta_total"] + BASE_VARS
for col in needed:
    if col not in df.columns:
        raise ValueError(f"Required column missing: {col}")

all_test_vars = [c for c in CANDIDATE_VARS if c in df.columns and c not in BASE_VARS]

for c in ["delta_total"] + BASE_VARS + all_test_vars:
    df[c] = pd.to_numeric(df[c], errors="coerce")

results = []

base_df = df[["delta_total"] + BASE_VARS].dropna().copy()
base_model = sm.OLS(
    base_df["delta_total"],
    sm.add_constant(base_df[BASE_VARS])
).fit()

print("Baseline model:")
print(f"delta_total ~ {' + '.join(BASE_VARS)}")
print(f"n = {int(base_model.nobs)}")
print(f"R2 = {base_model.rsquared:.4f}")
print(f"Adj R2 = {base_model.rsquared_adj:.4f}")
print(f"AIC = {base_model.aic:.2f}")
print()

for var in all_test_vars:
    sub = df[["delta_total"] + BASE_VARS + [var]].dropna().copy()
    if len(sub) < 5:
        continue

    m1 = sm.OLS(
        sub["delta_total"],
        sm.add_constant(sub[BASE_VARS])
    ).fit()

    m2 = sm.OLS(
        sub["delta_total"],
        sm.add_constant(sub[BASE_VARS + [var]])
    ).fit()

    f_stat, p_nested, _ = m2.compare_f_test(m1)

    coef = m2.params.get(var, np.nan)
    se = m2.bse.get(var, np.nan)
    tval = m2.tvalues.get(var, np.nan)
    pval = m2.pvalues.get(var, np.nan)
    ci_low, ci_high = m2.conf_int().loc[var].tolist()

    delta_r2 = m2.rsquared - m1.rsquared
    f2 = (delta_r2 / (1 - m2.rsquared)) if m2.rsquared < 1 else np.nan

    results.append({
        "variable": var,
        "n": int(m2.nobs),
        "base_r2": m1.rsquared,
        "full_r2": m2.rsquared,
        "delta_r2": delta_r2,
        "f2_effect_size": f2,
        "coef": coef,
        "std_err": se,
        "t_value": tval,
        "p_value": pval,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "nested_model_p_value": p_nested,
        "nested_f_stat": f_stat,
        "aic_base": m1.aic,
        "aic_full": m2.aic,
        "aic_improvement": m1.aic - m2.aic,
    })

res = pd.DataFrame(results)

if not res.empty:
    res = res.sort_values(
        by=["nested_model_p_value", "delta_r2"],
        ascending=[True, False]
    )

res.to_csv(OUT_CSV, index=False)

better = res[
    (res["nested_model_p_value"].notna()) &
    (res["nested_model_p_value"] < ALPHA)
].copy()

print(f"Saved -> {OUT_CSV}")
print()
print("Variables that add information beyond Length + Average_dSASA:")
if better.empty:
    print("None at p < 0.05")
else:
    print(
        better[[
            "variable",
            "n",
            "delta_r2",
            "f2_effect_size",
            "coef",
            "std_err",
            "ci_low",
            "ci_high",
            "p_value",
            "nested_model_p_value",
            "aic_improvement",
        ]].to_string(index=False)
    )

Baseline model:
delta_total ~ Length + Average_dSASA
n = 351
R2 = 0.7867
Adj R2 = 0.7855
AIC = 2336.67



/FastHome/gyula/anaconda3/envs/gyula_env/lib/python3.14/site-packages/statsmodels/regression/linear_model.py:2284: RuntimeWarning: divide by zero encountered in scalar divide
  f_value = (ssr_restr - ssr_full) / df_diff / ssr_full * df_full
/FastHome/gyula/anaconda3/envs/gyula_env/lib/python3.14/site-packages/statsmodels/regression/linear_model.py:2284: RuntimeWarning: divide by zero encountered in scalar divide
  f_value = (ssr_restr - ssr_full) / df_diff / ssr_full * df_full
/FastHome/gyula/anaconda3/envs/gyula_env/lib/python3.14/site-packages/statsmodels/regression/linear_model.py:2284: RuntimeWarning: divide by zero encountered in scalar divide
  f_value = (ssr_restr - ssr_full) / df_diff / ssr_full * df_full


Saved -> /FastHome/gyula/designs/place_holder/DATA/gy_5/output_new/delta_total_linear_model_vs_length_dsasa.csv

Variables that add information beyond Length + Average_dSASA:
                    variable   n  delta_r2  f2_effect_size       coef   std_err     ci_low    ci_high  p_value  nested_model_p_value  aic_improvement
               Average_i_pTM 351  0.008490        0.041456 -60.754491 16.018460 -92.259984 -29.248999 0.000176              0.000176        12.257474
            Average_ss_pLDDT 351  0.008381        0.040900 -54.685159 14.515874 -83.235328 -26.134990 0.000194              0.000194        12.070083
               Average_pLDDT 351  0.007769        0.037803 -41.422314 11.436876 -63.916637 -18.927992 0.000336              0.000336        11.024144
             Average_i_pLDDT 351  0.005727        0.027590 -39.833933 12.873973 -65.154772 -14.513094 0.002134              0.002134         7.552906
         Average_Binder_RMSD 351  0.005572        0.026826   0.823509  0.26

In [37]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path

INPUT_CSV = "/FastHome/gyula/designs/place_holder/DATA/gy_5/mpnn_design_stats_augmented.csv"
OUTPUT_DIR = Path("/FastHome/gyula/designs/place_holder/DATA/gy_5/output_new")
OUTPUT_DIR.mkdir(exist_ok=True)
OUT_CSV = OUTPUT_DIR / "delta_total_linear_model_vs_dsasa_i_pTM.csv"

BASE_VARS = [ "Average_dSASA", "Average_i_pTM" ]

CANDIDATE_VARS = [
    "Length",
    "Helicity",
    "MPNN_score",
    "MPNN_seq_recovery",
    "Average_pLDDT",
    "Average_pTM",
    "Average_i_pTM",
    "Average_pAE",
    "Average_i_pAE",
    "Average_i_pLDDT",
    "Average_ss_pLDDT",
    "Average_Unrelaxed_Clashes",
    "Average_Relaxed_Clashes",
    "Average_Binder_Energy_Score",
    "Average_Surface_Hydrophobicity",
    "Average_ShapeComplementarity",
    "Average_PackStat",
    "Average_dG",
    "Average_dG/dSASA",
    "Average_Interface_SASA_%",
    "Average_Interface_Hydrophobicity",
    "Average_n_InterfaceResidues",
    "Average_n_InterfaceHbonds",
    "Average_InterfaceHbondsPercentage",
    "Average_n_InterfaceUnsatHbonds",
    "Average_InterfaceUnsatHbondsPercentage",
    "Average_Interface_Helix%",
    "Average_Interface_BetaSheet%",
    "Average_Interface_Loop%",
    "Average_Binder_Helix%",
    "Average_Binder_BetaSheet%",
    "Average_Binder_Loop%",
    "Average_Hotspot_RMSD",
    "Average_Target_RMSD",
    "Average_Binder_pLDDT",
    "Average_Binder_pTM",
    "Average_Binder_pAE",
    "Average_Binder_RMSD",
]

ALPHA = 0.05

df = pd.read_csv(INPUT_CSV).copy()

if "delta_total" not in df.columns:
    raise ValueError("delta_total column not found")

needed = ["delta_total"] + BASE_VARS
for col in needed:
    if col not in df.columns:
        raise ValueError(f"Required column missing: {col}")

all_test_vars = [c for c in CANDIDATE_VARS if c in df.columns and c not in BASE_VARS]

for c in ["delta_total"] + BASE_VARS + all_test_vars:
    df[c] = pd.to_numeric(df[c], errors="coerce")

results = []

base_df = df[["delta_total"] + BASE_VARS].dropna().copy()
base_model = sm.OLS(
    base_df["delta_total"],
    sm.add_constant(base_df[BASE_VARS])
).fit()

print("Baseline model:")
print(f"delta_total ~ {' + '.join(BASE_VARS)}")
print(f"n = {int(base_model.nobs)}")
print(f"R2 = {base_model.rsquared:.4f}")
print(f"Adj R2 = {base_model.rsquared_adj:.4f}")
print(f"AIC = {base_model.aic:.2f}")
print()

for var in all_test_vars:
    sub = df[["delta_total"] + BASE_VARS + [var]].dropna().copy()
    if len(sub) < 5:
        continue

    m1 = sm.OLS(
        sub["delta_total"],
        sm.add_constant(sub[BASE_VARS])
    ).fit()

    m2 = sm.OLS(
        sub["delta_total"],
        sm.add_constant(sub[BASE_VARS + [var]])
    ).fit()

    f_stat, p_nested, _ = m2.compare_f_test(m1)

    coef = m2.params.get(var, np.nan)
    se = m2.bse.get(var, np.nan)
    tval = m2.tvalues.get(var, np.nan)
    pval = m2.pvalues.get(var, np.nan)
    ci_low, ci_high = m2.conf_int().loc[var].tolist()

    delta_r2 = m2.rsquared - m1.rsquared
    f2 = (delta_r2 / (1 - m2.rsquared)) if m2.rsquared < 1 else np.nan

    results.append({
        "variable": var,
        "n": int(m2.nobs),
        "base_r2": m1.rsquared,
        "full_r2": m2.rsquared,
        "delta_r2": delta_r2,
        "f2_effect_size": f2,
        "coef": coef,
        "std_err": se,
        "t_value": tval,
        "p_value": pval,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "nested_model_p_value": p_nested,
        "nested_f_stat": f_stat,
        "aic_base": m1.aic,
        "aic_full": m2.aic,
        "aic_improvement": m1.aic - m2.aic,
    })

res = pd.DataFrame(results)

if not res.empty:
    res = res.sort_values(
        by=["nested_model_p_value", "delta_r2"],
        ascending=[True, False]
    )

res.to_csv(OUT_CSV, index=False)

better = res[
    (res["nested_model_p_value"].notna()) &
    (res["nested_model_p_value"] < ALPHA)
].copy()

print(f"Saved -> {OUT_CSV}")
print()
print("Variables that add information beyond Length + Average_dSASA:")
if better.empty:
    print("None at p < 0.05")
else:
    print(
        better[[
            "variable",
            "n",
            "delta_r2",
            "f2_effect_size",
            "coef",
            "std_err",
            "ci_low",
            "ci_high",
            "p_value",
            "nested_model_p_value",
            "aic_improvement",
        ]].to_string(index=False)
    )

Baseline model:
delta_total ~ Average_dSASA + Average_i_pTM
n = 351
R2 = 0.7951
Adj R2 = 0.7940
AIC = 2322.51



/FastHome/gyula/anaconda3/envs/gyula_env/lib/python3.14/site-packages/statsmodels/regression/linear_model.py:2284: RuntimeWarning: divide by zero encountered in scalar divide
  f_value = (ssr_restr - ssr_full) / df_diff / ssr_full * df_full
/FastHome/gyula/anaconda3/envs/gyula_env/lib/python3.14/site-packages/statsmodels/regression/linear_model.py:2284: RuntimeWarning: divide by zero encountered in scalar divide
  f_value = (ssr_restr - ssr_full) / df_diff / ssr_full * df_full
/FastHome/gyula/anaconda3/envs/gyula_env/lib/python3.14/site-packages/statsmodels/regression/linear_model.py:2284: RuntimeWarning: divide by zero encountered in scalar divide
  f_value = (ssr_restr - ssr_full) / df_diff / ssr_full * df_full


Saved -> /FastHome/gyula/designs/place_holder/DATA/gy_5/output_new/delta_total_linear_model_vs_dsasa_i_pTM.csv

Variables that add information beyond Length + Average_dSASA:
                    variable   n  delta_r2  f2_effect_size       coef  std_err     ci_low   ci_high  p_value  nested_model_p_value  aic_improvement
         Average_Binder_RMSD 351  0.003783        0.018814   0.686253 0.268582   0.158000  1.214507 0.011042              0.011042         4.542429
Average_ShapeComplementarity 351  0.002719        0.013453 -19.556868 9.051483 -37.359542 -1.754194 0.031408              0.031408         2.690635


Since ipTM has a nonlinear relationship, lets try to squere it.

In [49]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path

INPUT_CSV = "/FastHome/gyula/designs/place_holder/DATA/gy_5/mpnn_design_stats_augmented.csv"
OUTPUT_DIR = Path("/FastHome/gyula/designs/place_holder/DATA/gy_5/output_new")
OUTPUT_DIR.mkdir(exist_ok=True)
OUT_CSV = OUTPUT_DIR / "delta_total_linear_model_vs_dsasa_i_pTM.csv"

BASE_VARS = [  "Average_i_pTM_sqr" ]

CANDIDATE_VARS = [

    "Length",
    "Helicity",
    "MPNN_score",
    "MPNN_seq_recovery",
    "Average_pLDDT",
    "Average_pTM",
    "Average_i_pTM",
    "Average_pAE",
    "Average_i_pAE",
    "Average_i_pLDDT",
    "Average_ss_pLDDT",
    "Average_Unrelaxed_Clashes",
    "Average_Relaxed_Clashes",
    "Average_Binder_Energy_Score",
    "Average_Surface_Hydrophobicity",
    "Average_ShapeComplementarity",
    "Average_PackStat",
    "Average_dG",
    "Average_dG/dSASA",
    "Average_Interface_SASA_%",
    "Average_Interface_Hydrophobicity",
    "Average_n_InterfaceResidues",
    "Average_n_InterfaceHbonds",
    "Average_InterfaceHbondsPercentage",
    "Average_n_InterfaceUnsatHbonds",
    "Average_InterfaceUnsatHbondsPercentage",
    "Average_Interface_Helix%",
    "Average_Interface_BetaSheet%",
    "Average_Interface_Loop%",
    "Average_Binder_Helix%",
    "Average_Binder_BetaSheet%",
    "Average_Binder_Loop%",
    "Average_Hotspot_RMSD",
    "Average_Target_RMSD",
    "Average_Binder_pLDDT",
    "Average_Binder_pTM",
    "Average_Binder_pAE",
    "Average_Binder_RMSD",
]

ALPHA = 0.05

df = pd.read_csv(INPUT_CSV).copy()

df["Average_i_pTM_sqr"] =  np.exp(df["Average_i_pTM"])


if "delta_total" not in df.columns:
    raise ValueError("delta_total column not found")

needed = ["delta_total"] + BASE_VARS
for col in needed:
    if col not in df.columns:
        raise ValueError(f"Required column missing: {col}")

all_test_vars = [c for c in CANDIDATE_VARS if c in df.columns and c not in BASE_VARS]

for c in ["delta_total"] + BASE_VARS + all_test_vars:
    df[c] = pd.to_numeric(df[c], errors="coerce")

results = []

base_df = df[["delta_total"] + BASE_VARS].dropna().copy()
base_model = sm.OLS(
    base_df["delta_total"],
    sm.add_constant(base_df[BASE_VARS])
).fit()

print("Baseline model:")
print(f"delta_total ~ {' + '.join(BASE_VARS)}")
print(f"n = {int(base_model.nobs)}")
print(f"R2 = {base_model.rsquared:.4f}")
print(f"Adj R2 = {base_model.rsquared_adj:.4f}")
print(f"AIC = {base_model.aic:.2f}")
print()

for var in all_test_vars:
    sub = df[["delta_total"] + BASE_VARS + [var]].dropna().copy()
    if len(sub) < 5:
        continue

    m1 = sm.OLS(
        sub["delta_total"],
        sm.add_constant(sub[BASE_VARS])
    ).fit()

    m2 = sm.OLS(
        sub["delta_total"],
        sm.add_constant(sub[BASE_VARS + [var]])
    ).fit()

    f_stat, p_nested, _ = m2.compare_f_test(m1)

    coef = m2.params.get(var, np.nan)
    se = m2.bse.get(var, np.nan)
    tval = m2.tvalues.get(var, np.nan)
    pval = m2.pvalues.get(var, np.nan)
    ci_low, ci_high = m2.conf_int().loc[var].tolist()

    delta_r2 = m2.rsquared - m1.rsquared
    f2 = (delta_r2 / (1 - m2.rsquared)) if m2.rsquared < 1 else np.nan

    results.append({
        "variable": var,
        "n": int(m2.nobs),
        "base_r2": m1.rsquared,
        "full_r2": m2.rsquared,
        "delta_r2": delta_r2,
        "f2_effect_size": f2,
        "coef": coef,
        "std_err": se,
        "t_value": tval,
        "p_value": pval,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "nested_model_p_value": p_nested,
        "nested_f_stat": f_stat,
        "aic_base": m1.aic,
        "aic_full": m2.aic,
        "aic_improvement": m1.aic - m2.aic,
    })

res = pd.DataFrame(results)

if not res.empty:
    res = res.sort_values(
        by=["nested_model_p_value", "delta_r2"],
        ascending=[True, False]
    )

res.to_csv(OUT_CSV, index=False)

better = res[
    (res["nested_model_p_value"].notna()) &
    (res["nested_model_p_value"] < ALPHA)
].copy()

print(f"Saved -> {OUT_CSV}")
print()
print("Variables that add information beyond Length + Average_dSASA:")
if better.empty:
    print("None at p < 0.05")
else:
    print(
        better[[
            "variable",
            "n",
            "delta_r2",
            "f2_effect_size",
            "coef",
            "std_err",
            "ci_low",
            "ci_high",
            "p_value",
            "nested_model_p_value",
            "aic_improvement",
        ]].to_string(index=False)
    )

Baseline model:
delta_total ~ Average_i_pTM_sqr
n = 351
R2 = 0.1273
Adj R2 = 0.1248
AIC = 2829.21



/FastHome/gyula/anaconda3/envs/gyula_env/lib/python3.14/site-packages/statsmodels/regression/linear_model.py:2284: RuntimeWarning: invalid value encountered in scalar divide
  f_value = (ssr_restr - ssr_full) / df_diff / ssr_full * df_full
/FastHome/gyula/anaconda3/envs/gyula_env/lib/python3.14/site-packages/statsmodels/regression/linear_model.py:2284: RuntimeWarning: invalid value encountered in scalar divide
  f_value = (ssr_restr - ssr_full) / df_diff / ssr_full * df_full
/FastHome/gyula/anaconda3/envs/gyula_env/lib/python3.14/site-packages/statsmodels/regression/linear_model.py:2284: RuntimeWarning: invalid value encountered in scalar divide
  f_value = (ssr_restr - ssr_full) / df_diff / ssr_full * df_full


Saved -> /FastHome/gyula/designs/place_holder/DATA/gy_5/output_new/delta_total_linear_model_vs_dsasa_i_pTM.csv

Variables that add information beyond Length + Average_dSASA:
                              variable   n  delta_r2  f2_effect_size         coef     std_err        ci_low      ci_high      p_value  nested_model_p_value  aic_improvement
           Average_n_InterfaceResidues 351  0.552993        1.729666    -4.097972    0.167031     -4.426490    -3.769453 7.017012e-78          7.017012e-78       350.466876
                                Length 351  0.544104        1.655830    -2.991382    0.124616     -3.236478    -2.746287 8.352797e-76          8.352797e-76       340.841810
                            Average_dG 351  0.457317        1.100942     1.033488    0.052800      0.929641     1.137335 4.677278e-58          4.677278e-58       258.577436
                    Average_Binder_pTM 351  0.376068        0.757233   -81.988137    5.050643    -91.921763   -72.054512 1.628214e-44 